# Tier 1A Model Workbench

This notebook trains and inspects the Tier 1A baseline models on `/data/processed/training_matrix_tier1a.csv`.

Included sections:
- data load and sanity checks
- company-grouped train/validation/test split
- Logistic Regression
- Random Forest
- HistGradientBoosting
- Isolation Forest
- KMeans clustering
- simple Tier 1A risk scorecard


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "training_matrix_tier1a.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "training_matrix_tier1a.csv"
assert DATA_PATH.exists(), f"Could not find {DATA_PATH}"

FEATURE_COLUMNS = [
    "debt_to_assets",
    "ebitda_margin",
    "pat_margin",
    "roa",
    "retained_earnings_to_assets",
    "cfo_to_assets",
    "cfo_to_ebitda",
    "net_cash_change_to_assets",
    "log_total_assets",
    "log_revenue",
]

ID_COLUMNS = ["company_name", "cin", "financial_year", "cohort", "sector"]
LABEL_COLUMN = "target_wilful_default"


In [ ]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("companies:", df["cin"].nunique())
print("target balance:")
print(df[LABEL_COLUMN].value_counts())
print()
print("feature coverage:")
coverage = pd.DataFrame({
    "present": df[FEATURE_COLUMNS].notna().sum(),
    "missing": df[FEATURE_COLUMNS].isna().sum(),
    "pct_present": (df[FEATURE_COLUMNS].notna().mean() * 100).round(2),
})
coverage


In [ ]:
def split_by_company(df, seed=42, train_companies=70, val_companies=15, test_companies=15):
    company_df = df[["cin", "company_name", LABEL_COLUMN]].drop_duplicates(subset=["cin"]).reset_index(drop=True)
    total = len(company_df)
    assert train_companies + val_companies + test_companies == total, (train_companies, val_companies, test_companies, total)

    train_comp, temp_comp = train_test_split(
        company_df,
        train_size=train_companies,
        stratify=company_df[LABEL_COLUMN],
        random_state=seed,
    )

    val_fraction = val_companies / (val_companies + test_companies)
    val_comp, test_comp = train_test_split(
        temp_comp,
        train_size=val_fraction,
        stratify=temp_comp[LABEL_COLUMN],
        random_state=seed,
    )

    train_df = df[df["cin"].isin(train_comp["cin"])].copy()
    val_df = df[df["cin"].isin(val_comp["cin"])].copy()
    test_df = df[df["cin"].isin(test_comp["cin"])].copy()
    return train_df, val_df, test_df

train_df, val_df, test_df = split_by_company(df)
for name, part in {"train": train_df, "validation": val_df, "test": test_df}.items():
    print(name, part.shape, part["cin"].nunique(), part[LABEL_COLUMN].value_counts().to_dict())


In [ ]:
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[LABEL_COLUMN].copy()
X_val = val_df[FEATURE_COLUMNS].copy()
y_val = val_df[LABEL_COLUMN].copy()
X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[LABEL_COLUMN].copy()

def evaluate_classifier(name, model, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train, y_train)
    out = []
    for split_name, X, y in [("validation", X_val, y_val), ("test", X_test, y_test)]:
        proba = model.predict_proba(X)[:, 1]
        pred = (proba >= 0.5).astype(int)
        out.append({
            "model": name,
            "split": split_name,
            "rows": len(X),
            "accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, proba),
            "pr_auc": average_precision_score(y, proba),
            "brier": brier_score_loss(y, proba),
            "confusion_matrix": confusion_matrix(y, pred).tolist(),
        })
    return model, pd.DataFrame(out)

logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(C=1.0, class_weight="balanced", solver="liblinear", max_iter=2000, random_state=42)),
])

random_forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )),
])

hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        random_state=42,
    )),
])

trained = {}
frames = []
for name, model in {
    "logistic_regression": logistic,
    "random_forest": random_forest,
    "hist_gradient_boosting": hgb,
}.items():
    fitted, metrics_df = evaluate_classifier(name, model, X_train, y_train, X_val, y_val, X_test, y_test)
    trained[name] = fitted
    frames.append(metrics_df)

metrics_table = pd.concat(frames, ignore_index=True)
metrics_table


In [ ]:
def top_permutation_importance(model_name, estimator, X, y, n_repeats=20):
    result = permutation_importance(estimator, X, y, n_repeats=n_repeats, random_state=42, scoring="average_precision")
    imp = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)
    print(model_name)
    return imp

perm_tables = {}
for name, est in trained.items():
    perm_tables[name] = top_permutation_importance(name, est, X_val, y_val)
    display(perm_tables[name].head(10))


In [ ]:
iso_imputer = SimpleImputer(strategy="median")
iso_scaler = StandardScaler()
X_unsup = iso_scaler.fit_transform(iso_imputer.fit_transform(df[FEATURE_COLUMNS]))

iso = IsolationForest(n_estimators=300, contamination=0.15, random_state=42)
iso.fit(X_unsup)

df_iso = df[ID_COLUMNS + [LABEL_COLUMN]].copy()
df_iso["anomaly_score"] = -iso.decision_function(X_unsup)
df_iso["anomaly_flag"] = (iso.predict(X_unsup) == -1).astype(int)

print("Anomaly summary by target:")
summary = df_iso.groupby(LABEL_COLUMN).agg(
    mean_score=("anomaly_score", "mean"),
    flagged_rate=("anomaly_flag", "mean"),
    rows=("anomaly_flag", "size"),
)
summary


In [ ]:
kmeans_imputer = SimpleImputer(strategy="median")
kmeans_scaler = StandardScaler()
X_cluster = kmeans_scaler.fit_transform(kmeans_imputer.fit_transform(df[FEATURE_COLUMNS]))

cluster_model = KMeans(n_clusters=4, random_state=42, n_init=20)
clusters = cluster_model.fit_predict(X_cluster)

df_cluster = df[ID_COLUMNS + [LABEL_COLUMN]].copy()
df_cluster["cluster"] = clusters

cluster_summary = df_cluster.groupby("cluster").agg(
    rows=("cluster", "size"),
    default_rate=(LABEL_COLUMN, "mean"),
    companies=("cin", "nunique"),
)
print("Cluster summary:")
cluster_summary


In [ ]:
score_df = df[ID_COLUMNS + [LABEL_COLUMN] + FEATURE_COLUMNS].copy()

thresholds = {
    "debt_to_assets_hi": score_df["debt_to_assets"].quantile(0.75),
    "ebitda_margin_lo": score_df["ebitda_margin"].quantile(0.25),
    "pat_margin_lo": score_df["pat_margin"].quantile(0.25),
    "roa_lo": score_df["roa"].quantile(0.25),
    "retained_earnings_to_assets_lo": score_df["retained_earnings_to_assets"].quantile(0.25),
    "cfo_to_assets_lo": score_df["cfo_to_assets"].quantile(0.25),
    "cfo_to_ebitda_lo": score_df["cfo_to_ebitda"].quantile(0.25),
    "net_cash_change_to_assets_lo": score_df["net_cash_change_to_assets"].quantile(0.25),
}

score_df["leverage_flag"] = (score_df["debt_to_assets"] >= thresholds["debt_to_assets_hi"]).astype(int)
score_df["ebitda_flag"] = (score_df["ebitda_margin"] <= thresholds["ebitda_margin_lo"]).astype(int)
score_df["pat_flag"] = (score_df["pat_margin"] <= thresholds["pat_margin_lo"]).astype(int)
score_df["roa_flag"] = (score_df["roa"] <= thresholds["roa_lo"]).astype(int)
score_df["retained_flag"] = (score_df["retained_earnings_to_assets"] <= thresholds["retained_earnings_to_assets_lo"]).astype(int)
score_df["cfo_asset_flag"] = (score_df["cfo_to_assets"] <= thresholds["cfo_to_assets_lo"]).astype(int)
score_df["cfo_ebitda_flag"] = (score_df["cfo_to_ebitda"] <= thresholds["cfo_to_ebitda_lo"]).astype(int)
score_df["cash_change_flag"] = (score_df["net_cash_change_to_assets"] <= thresholds["net_cash_change_to_assets_lo"]).astype(int)

flag_cols = [c for c in score_df.columns if c.endswith("_flag")]
score_df["risk_score_tier1a"] = (score_df[flag_cols].sum(axis=1) / len(flag_cols) * 100).round(2)

print("Scorecard summary by target:")
scorecard_summary = score_df.groupby(LABEL_COLUMN).agg(
    mean_score=("risk_score_tier1a", "mean"),
    median_score=("risk_score_tier1a", "median"),
    rows=("risk_score_tier1a", "size"),
)
scorecard_summary


## Recommended next step

Run this notebook first as the baseline. After that, compare against Tier 1B and then add the annual-report features as they become available.
